# 网络健全性检查

从零构建默认网络，检查潮流收敛、电压范围与功率平衡，并导出母线电压等级分布图。

In [1]:
from pathlib import Path


def repo_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "configs" / "default.yaml").exists():
            return candidate
    raise FileNotFoundError("未找到仓库根目录")

ROOT = repo_root()

In [2]:
from ltverify.config import load_config
from ltverify.network import build_network, topology_frames

cfg = load_config(ROOT / "configs/default.yaml")
artifacts = build_network(cfg.network)
print("母线:", len(artifacts.net.bus), "| 线路:", len(artifacts.net.line))
print("配变:", len(artifacts.asset_table), "| 馈线数:", artifacts.asset_table["physical_feeder_id"].nunique())
print("光伏数量:", artifacts.asset_table["pv_index"].notna().sum())

母线:

53

| 线路:

27

配变:

24

| 馈线数:

3

光伏数量:

6

In [3]:
from ltverify.validation import run_static_validation

static = run_static_validation(artifacts, cfg.validation)
print("收敛:", static.converged)
print("电压范围(p.u.):", round(static.voltage_min_pu, 4), "-", round(static.voltage_max_pu, 4))
print("功率平衡误差(MW):", static.absolute_power_balance_error_mw)
print("违规项:", static.violations)

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


收敛:

True

电压范围(p.u.):

1.0

...

1.0001

功率平衡误差(MW):

7.459819554250894e-14

违规项:

()

In [4]:
import plotly.graph_objects as go

nodes, edges = topology_frames(artifacts)
figure = go.Figure()
figure.add_trace(go.Scatter(x=nodes["node_id"], y=nodes["voltage_kv"], mode="markers", name="母线"))
figure.update_layout(title="01 网络母线电压等级分布", xaxis_title="节点编号", yaxis_title="电压等级 (kV)")
output = ROOT / "reports/figures/01_network_sanity.html"
output.parent.mkdir(parents=True, exist_ok=True)
figure.write_html(output)
print("图已保存:", output.resolve())

图已保存:

D:\电力\reports\figures\01_network_sanity.html